In [9]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import StandardScaler
import seaborn as sns
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from sklearn.pipeline import Pipeline
import joblib
from sklearn.preprocessing import StandardScaler
from scipy.stats import pearsonr
from scipy.stats import  chi2_contingency
import warnings

warnings.filterwarnings("ignore")

# Load data
df = pd.read_csv("insurance.csv")

# Remove duplicates
df = df.drop_duplicates()

# Convert categorical columns to numbers
df["sex"] = df["sex"].map({"male": 0, "female": 1})
df["smoker"] = df["smoker"].map({"yes": 1, "no": 0})

# Rename columns
df.rename(columns={
    "smoker": "issmoker",
    "sex": "isfemale"
}, inplace=True)

# One-hot encode region
df = pd.get_dummies(df, columns=["region"])

# Convert existing columns to integers
df = df.astype(int)

# -----------------------------
# CREATE GROUPS
# -----------------------------

# Age groups
df["age_group"] = pd.cut(
    df["age"],
    bins=[17, 29, 44, 59, float("inf")],
    labels=[
        "Young Adult",
        "Adult",
        "Middle-aged Adult",
        "Older Adult"
    ]
)

df = pd.get_dummies(
    df,
    columns=["age_group"],
    dtype=int
)


# BMI groups
df["bmi_group"] = pd.cut(
    df["bmi"],
    bins=[0, 18.49, 24.9, 28.9, 34.9, 38.9, float("inf")],
    labels=[
        "underweight",
        "normal",
        "overweight",
        "obese 1",
        "obese II",
        "obese III"
        
    ]
)

# Expenses groups

# One-hot encode BMI groups
df = pd.get_dummies(df, columns=["bmi_group"],dtype=int)

# -----------------------------
# CORRELATION
# -----------------------------

col = [
    "age",
    "isfemale",
    "bmi",
    "children",
    "issmoker",
    "region_northeast",
    "region_northwest",
    "region_southeast",
    "region_southwest",
    "age_group_Young Adult",
    "age_group_Adult",
    "age_group_Middle-aged Adult",
    "age_group_Older Adult",
    "bmi_group_underweight",
    "bmi_group_normal",
    "bmi_group_overweight",
    "bmi_group_obese 1",
    "bmi_group_obese II",
    "bmi_group_obese III"
]
# -----------------------------
# PEARSON CORRELATION
# -----------------------------

correlations = {
    feature: pearsonr(df[feature], df["expenses"])
    for feature in col
}

# Convert dictionary to DataFrame
correlation_df = pd.DataFrame(
    [
        [feature, result[0], result[1]]
        for feature, result in correlations.items()
    ],
    columns=["Feature", "Pearson Correlation", "p_value"]
)

# Sort from highest to lowest correlation
correlation_df = correlation_df.sort_values(
    by="Pearson Correlation",
    ascending=False
)

# Show statistically significant correlations
sg = correlation_df.loc[
    correlation_df["p_value"] < 0.05,
    ["Feature", "Pearson Correlation", "p_value"]]
sg


,Feature,Pearson Correlation,p_value
4,issmoker,0.787234,1.409368e-282
0,age,0.298309,6.974034e-29
12,age_group_Older Adult,0.200976,1.192499e-13
2,bmi,0.197714,2.991509e-13
17,bmi_group_obese II,0.159490,4.521885e-09
11,age_group_Middle-aged Adult,0.146519,7.385826e-08
7,region_southeast,0.073577,7.113282e-03
3,children,0.067390,1.371568e-02
18,bmi_group_obese III,0.064531,1.828348e-02
1,isfemale,-0.058046,3.381620e-02


In [10]:
cols = [
    "issmoker",
    "age",
    "age_group_Older Adult",
    "bmi",
    "bmi_group_obese II",
    "age_group_Middle-aged Adult",
    "region_southeast",
    "children",
    "bmi_group_obese III",
    "isfemale",
    "bmi_group_overweight",
    "bmi_group_normal",
    "age_group_Young Adult"
]


In [11]:
models = {
    "Model 1 - Original": [
        "age",
        "bmi",
        "isfemale",
        "issmoker",
        "children",
     
        "region_southeast"
    ],

    "Model 2 - Age Groups": [
        "age_group_Young Adult",
        "age_group_Middle-aged Adult",
        "age_group_Older Adult",
        "bmi",
        "isfemale",
        "issmoker",
        "children",
      
        "region_southeast"
    ],

    "Model 3 - BMI Groups": [
        "age",
        "bmi_group_underweight",
        "bmi_group_normal",
        "bmi_group_overweight",
        "bmi_group_obese II",
        "bmi_group_obese III",
        "isfemale",
        "issmoker",
        "children",

        "region_southeast"
    ],

    "Model 4 - Both Groups": [
        "age_group_Young Adult",
        "age_group_Middle-aged Adult",
        "age_group_Older Adult",
        "bmi_group_underweight",
        "bmi_group_normal",
        "bmi_group_overweight",
        "bmi_group_obese II",
        "bmi_group_obese III",
        "isfemale",
        "issmoker",
        "children",
   
        "region_southeast"
    ]
}


In [12]:
results = []

for i, j in models.items():

    y = df["expenses"]
    x = df[j]

    X_train, X_test, y_train, y_test = train_test_split(
        x, y,
        test_size=0.2,
        random_state=42
    )

    sd = StandardScaler()

    X_train_scaled = sd.fit_transform(X_train)
    x_test_scaled = sd.transform(X_test)

    rd = Ridge(1)

    rd.fit(X_train_scaled, y_train)

    y_pred = rd.predict(x_test_scaled)
    y_train_pred = rd.predict(X_train_scaled)

    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_pred)

    r2 = test_r2

    n = len(y_test)
    p = X_test.shape[1]

    adjusted_r2 = 1 - (1-r2)*(n-1)/(n-p-1)

    mae = mean_absolute_error(y_test, y_pred)

    rmse = np.sqrt(
        mean_squared_error(y_test, y_pred)
    )

    results.append([
        i,
        train_r2,
        test_r2,
        adjusted_r2,
        mae,
        rmse
    ])

results_df = pd.DataFrame(
    results,
    columns=[
        "Model",
        "Train R²",
        "Test R²",
        "Adjusted R²",
        "MAE",
        "RMSE"
    ]
)

best_model = results_df.sort_values(
    by=["Test R²", "MAE", "RMSE", "Adjusted R²", "Train R²"],
    ascending=[False, True, True, False, False]
).iloc[0]

print("Best Model:", best_model["Model"])
print(best_model)

Best Model: Model 1 - Original
Model          Model 1 - Original
Train R²                 0.729432
Test R²                   0.80525
Adjusted R²              0.800773
MAE                   4186.469766
RMSE                  5982.186757
Name: 0, dtype: object


In [13]:

best_features=[   "age",
        "bmi",
        "isfemale",
        "issmoker",
        "children",
     
        "region_southeast"]
X = df[best_features]
y = df["expenses"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

final_model = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=1))
])

final_model.fit(X_train, y_train)

joblib.dump(final_model, "insurance_model.pkl")

['insurance_model.pkl']